# Step 6: Model Deployment

Deploy the trained model for inference. Snowflake supports **two deployment approaches**:

| Aspect | SQL Inference | REST Inference |
|--------|--------------|----------------|
| **How to Access** | SQL queries inside Snowflake | HTTP requests from external apps |
| **Best For** | Batch processing on tables | Real-time single predictions |
| **Latency** | Higher (warehouse startup) | Lower (always-on service) |
| **Use Case** | ETL pipelines, scheduled jobs | Web apps, mobile apps, APIs |
| **Scaling** | Warehouse size | SPCS auto-scaling |
| **Cost Model** | Warehouse credits | SPCS compute pool |

## Prerequisites

- Run notebooks 01-05 first

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import os
import sys
import time
import json
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from source.configs import get_config
from source.utils import get_session, get_feature_config

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Get Model Version from Registry

In [ ]:
MODEL_NAME = config.model.model_name

registry = Registry(
    session,
    database_name=DB,
    schema_name=SCHEMA,
)

model = registry.get_model(MODEL_NAME)
versions = model.versions()
MODEL_VERSION = versions[-1].version_name

print(f"Model: {MODEL_NAME}")
print(f"Available versions: {[v.version_name for v in versions]}")
print(f"Version to deploy: {MODEL_VERSION}")

In [ ]:
model_version = model.version(MODEL_VERSION)
print(f"Existing services: {model_version.list_services()}")

## Option 1: SQL Inference Deployment

Deploy the model for SQL inference by setting the default version. This allows calling `MODEL!PREDICT()` directly in SQL queries.

**Use when**: Batch processing on Snowflake tables, scheduled ETL jobs, data pipelines

### Set Default Model Version

In [ ]:
print(f"Setting {MODEL_NAME} version {MODEL_VERSION} as default...")
model.default = MODEL_VERSION

sql_endpoint = f"{DB}.{SCHEMA}.{MODEL_NAME}!PREDICT()"

print(f"Default version set: {MODEL_VERSION}")
print(f"SQL endpoint: {sql_endpoint}")
print(f"Deployed at: {datetime.now().isoformat()}")

### Test SQL Inference

Run inference using SQL on sample data from the test table.

In [ ]:
sql_query = f"""
SELECT 
    AGE,
    GENDER,
    ADMISSION_TYPE,
    RISK_LEVEL AS ACTUAL_RISK,
    {DB}.{SCHEMA}.{MODEL_NAME}!PREDICT(
        AGE => AGE, BMI => BMI, HEART_RATE => HEART_RATE,
        SYSTOLIC_BP => SYSTOLIC_BP, DIASTOLIC_BP => DIASTOLIC_BP,
        TEMPERATURE => TEMPERATURE, RESPIRATORY_RATE => RESPIRATORY_RATE,
        OXYGEN_SATURATION => OXYGEN_SATURATION, GLUCOSE_LEVEL => GLUCOSE_LEVEL,
        CREATININE => CREATININE, HEMOGLOBIN => HEMOGLOBIN, WBC_COUNT => WBC_COUNT,
        COMORBIDITY_COUNT => COMORBIDITY_COUNT, PREVIOUS_ADMISSIONS => PREVIOUS_ADMISSIONS,
        MEDICATION_COUNT => MEDICATION_COUNT, GENDER => GENDER,
        PRIMARY_DIAGNOSIS => PRIMARY_DIAGNOSIS, ADMISSION_TYPE => ADMISSION_TYPE,
        INSURANCE_TYPE => INSURANCE_TYPE, SHOCK_INDEX => SHOCK_INDEX,
        PULSE_PRESSURE => PULSE_PRESSURE, BMI_CATEGORY => BMI_CATEGORY,
        VITAL_SIGNS_SEVERITY => VITAL_SIGNS_SEVERITY
    ):output_feature_0::STRING AS PREDICTED_RISK
FROM {DB}.{SCHEMA}.TEST_FEATURES
LIMIT 5
"""

print("SQL Inference Query:")
print(sql_query)
print("\nResults:")
session.sql(sql_query).show()

## Option 2: REST Inference Deployment (SPCS Service)

Deploy the model as a containerized REST endpoint on Snowpark Container Services for real-time inference from external applications.

**Use when**: Web/mobile apps, external API integrations, low-latency real-time predictions

**Note**: Service creation takes 5-10 minutes on first deployment.

### Create Inference Service

In [ ]:
SERVICE_NAME = "PATIENT_RISK_SERVICE"
COMPUTE_POOL = config.compute.compute_pool
AUTOCAPTURE = True

print(f"Creating inference service: {SERVICE_NAME}")
print(f"  Model: {MODEL_NAME}, Version: {MODEL_VERSION}")
print(f"  Compute pool: {COMPUTE_POOL}")
print(f"  Autocapture: {AUTOCAPTURE}")

try:
    model_version.create_service(
        service_name=SERVICE_NAME,
        service_compute_pool=COMPUTE_POOL,
        image_build_compute_pool=COMPUTE_POOL,
        min_instances=1,
        max_instances=1,
        ingress_enabled=True,
        autocapture=AUTOCAPTURE,
    )
    print(f"Service creation initiated: {SERVICE_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Service {SERVICE_NAME} already exists")
    else:
        raise

### Wait for Service to be Ready

In [ ]:
TIMEOUT = 600
POLL_INTERVAL = 10

print(f"Waiting for service {SERVICE_NAME} to be ready (timeout: {TIMEOUT}s)...")

start_time = time.time()
last_status = None

while time.time() - start_time < TIMEOUT:
    result = session.sql(
        f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DB}.{SCHEMA}"
    ).collect()

    if result:
        current_status = result[0]["status"]
        if current_status != last_status:
            elapsed = int(time.time() - start_time)
            print(f"  [{elapsed}s] Status: {current_status}")
            last_status = current_status

        if current_status == "RUNNING":
            print(f"Service {SERVICE_NAME} is RUNNING!")
            break

        if current_status in ("FAILED", "DELETING", "DELETED"):
            print(f"Service entered {current_status} state")
            break

    time.sleep(POLL_INTERVAL)
else:
    print(f"Timeout after {TIMEOUT}s waiting for service")

### Get Service Endpoint

In [ ]:
DEPLOYED_SERVICE_NAME = f"{DB}.{SCHEMA}.{SERVICE_NAME}"
MAX_RETRIES = 12
RETRY_INTERVAL = 10

rest_endpoint = None

for attempt in range(MAX_RETRIES):
    try:
        services = model_version.list_services().to_dict(orient="records")
        for svc in services:
            if svc.get("name", "").upper() == DEPLOYED_SERVICE_NAME.upper():
                rest_endpoint = svc.get("inference_endpoint")
                if rest_endpoint:
                    break
    except Exception as e:
        print(f"  Attempt {attempt + 1}: {e}")

    if rest_endpoint:
        break

    if attempt < MAX_RETRIES - 1:
        print(f"  Endpoint not yet available, retrying in {RETRY_INTERVAL}s (attempt {attempt + 1}/{MAX_RETRIES})")
        time.sleep(RETRY_INTERVAL)

if rest_endpoint:
    print(f"REST endpoint: {rest_endpoint}")
    if AUTOCAPTURE:
        print(f"Inference table: TABLE(INFERENCE_TABLE('{MODEL_NAME}'))")
else:
    print("REST endpoint not available after retries")

### Test REST Inference

Run inference using the REST API with a sample patient record.

In [ ]:
import requests

test_patient = {
    "AGE": 65,
    "BMI": 28.5,
    "HEART_RATE": 88,
    "SYSTOLIC_BP": 145,
    "DIASTOLIC_BP": 92,
    "TEMPERATURE": 37.2,
    "RESPIRATORY_RATE": 18,
    "OXYGEN_SATURATION": 96.0,
    "GLUCOSE_LEVEL": 142.0,
    "CREATININE": 1.3,
    "HEMOGLOBIN": 13.5,
    "WBC_COUNT": 8.2,
    "COMORBIDITY_COUNT": 2,
    "PREVIOUS_ADMISSIONS": 1,
    "MEDICATION_COUNT": 5,
    "GENDER": "M",
    "PRIMARY_DIAGNOSIS": "I10",
    "ADMISSION_TYPE": "Elective",
    "INSURANCE_TYPE": "Medicare",
    "SHOCK_INDEX": round(88 / 145, 4),
    "PULSE_PRESSURE": 145 - 92,
    "BMI_CATEGORY": "OVERWEIGHT",
    "VITAL_SIGNS_SEVERITY": 0,
}

print("REST Inference Example:")
print(f"  Endpoint: {rest_endpoint}/predict")
print(f"  Input: 65yo male, elective admission, BP 145/92, glucose 142")

PAT_TOKEN = os.environ.get("SNOWFLAKE_TOKEN", None)
if PAT_TOKEN is None:
    PAT_TOKEN = input("Enter your Snowflake PAT token: ").strip()

headers = {
    "Authorization": f'Snowflake Token="{PAT_TOKEN}"',
    "Content-Type": "application/json",
}

payload = {
    "dataframe_records": [test_patient]
}

print("\nCalling REST endpoint...")
response = requests.post(
    f"https://{rest_endpoint}/predict",
    headers=headers,
    json=payload,
    timeout=30,
)

print(f"  Response Status: {response.status_code}")
if response.ok:
    print(f"  Predicted Risk Level: {response.json()}")
else:
    print(f"  Error: {response.text}")

## Check Service Status

In [ ]:
result = session.sql(
    f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DB}.{SCHEMA}"
).collect()

if result:
    row = result[0]
    print(f"Service: {row['name']}")
    print(f"  Status: {row['status']}")
    print(f"  Compute Pool: {row['compute_pool']}")
    print(f"  Current Instances: {row['current_instances']}")
    print(f"  Target Instances: {row['target_instances']}")
    print(f"  DNS: {row['dns_name']}")
else:
    print("Service not found")

## Next Step

Continue to **07_model_monitoring.ipynb**